In [1]:
"""
notebook.ipynb  ← See cuad_pipeline_notebook.py for the .py version.

cuad_pipeline_notebook.py
--------------------------
Jupyter-style notebook as a plain Python script.
Convert to .ipynb with:  jupytext --to notebook cuad_pipeline_notebook.py

Run the full pipeline interactively, inspect results, and run semantic search.
"""

'\nnotebook.ipynb  ← See cuad_pipeline_notebook.py for the .py version.\n\ncuad_pipeline_notebook.py\n--------------------------\nJupyter-style notebook as a plain Python script.\nConvert to .ipynb with:  jupytext --to notebook cuad_pipeline_notebook.py\n\nRun the full pipeline interactively, inspect results, and run semantic search.\n'

# CUAD Contract Analysis Pipeline
### LLM-Powered Clause Extraction & Summarisation
**Dataset**: Contract Understanding Atticus Dataset (CUAD)  
**Model**  : Groq / Llama-3  
**Bonus**  : Semantic search with sentence-transformers

## 0. Setup

In [2]:
import os
import json
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()                          # Load GROQ_API_KEY from .env
assert os.getenv("GROQ_API_KEY"), "Set GROQ_API_KEY in your .env file!"

DATA_DIR   = Path("data")             # Folder with CUADv1.json / train.json
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

print("✓ Environment ready")

✓ Environment ready


## 1. Data Loading & Preprocessing

In [3]:
from src.data_loader import load_all_cuad_data

contracts = load_all_cuad_data(DATA_DIR, max_contracts=50)
print(f"Loaded {len(contracts)} contracts")

# Inspect first contract
c = contracts[0]
print(f"\nContract ID : {c['contract_id']}")
print(f"Title       : {c['title']}")
print(f"Text length : {len(c['text'])} chars")
print(f"\nFirst 500 chars:\n{c['text'][:500]}")

Loaded 50 contracts

Contract ID : contract_0001
Title       : LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT
Text length : 38670 chars

First 500 chars:
EXHIBIT 10.6

 DISTRIBUTOR AGREEMENT

 THIS DISTRIBUTOR AGREEMENT (the "Agreement") is made by and between Electric City Corp., a Delaware corporation ("Company") and Electric City of Illinois LLC ("Distributor") this 7th day of September, 1999.

 RECITALS

 A. The Company's Business. The Company is presently engaged in the business of selling an energy efficiency device, which is referred to as an "Energy Saver" which may be improved or otherwise changed from its present composition (the "Produ


## 2. LLM-Powered Extraction & Summarisation

### 2a. Single-contract demo (fast sanity-check)

In [4]:
from src.llm_client import extract_clauses, summarize_contract

demo_contract = contracts[0]
print(f"Processing: {demo_contract['title']}\n")

clauses = extract_clauses(demo_contract["text"], few_shot=True)
print("=== CLAUSES ===")
for k, v in clauses.items():
    print(f"[{k.upper()}]\n{v}\n")

summary = summarize_contract(demo_contract["text"])
print("=== SUMMARY ===")
print(summary)

Processing: LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT

=== CLAUSES ===
[TERMINATION_CLAUSE]
Either party may terminate upon 30 days written notice; the Company may terminate immediately upon material breach.

[CONFIDENTIALITY_CLAUSE]
Not specified in this contract.

[LIABILITY_CLAUSE]
Neither party is liable for indirect, incidental, or consequential damages arising from this agreement.

=== SUMMARY ===
This is a Distributor Agreement between Electric City Corp. and Electric City of Illinois LLC, where the latter is appointed as the exclusive distributor of Electric City Corp.'s energy efficiency device, the "Energy Saver", within the State of Illinois. Key obligations of each party include Electric City Corp.'s obligation to sell and deliver the Energy Saver to Electric City of Illinois LLC on agreed-upon terms, and the latter's obligation to promote the distribution, sales, and use of the Energy Saver in the Market. Notable risks and conditions include the possibility of El

### 2b. Full batch processing (all 50 contracts)
⚠️ This takes ~5–10 minutes for 50 contracts due to Groq rate limits.

In [5]:
from src.extractor import process_batch

results = process_batch(
    contracts,
    model="llama3-8b-8192",
    few_shot=True,
    output_dir=OUTPUT_DIR,
    resume=True,           # Skip already-processed contracts
)

print(f"\nTotal processed: {len(results)}")
success = sum(1 for r in results if r.get("status") == "success")
print(f"Success: {success} / {len(results)}")


Total processed: 50
Success: 50 / 50


## 3. Explore Results

In [6]:
df = pd.read_csv(OUTPUT_DIR / "results.csv")
df.head(3)

,contract_id,title,summary,termination_clause,confidentiality_clause,liability_clause,status,error
0,contract_0001,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,This is a Distributor Agreement between Electr...,Either party may terminate this Distributor Ag...,Not specified in this contract.,Neither party is liable for consequential or l...,success,NaN
1,contract_0002,"WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...",This is a Promotion and Distribution Agreement...,The agreement may be terminated by either part...,Confidential treatment has been requested for ...,Neither party is liable for indirect or conseq...,success,NaN
2,contract_0003,LohaCompanyltd_20191209_F-1_EX-10.16_11917878_...,This is a supply contract between Shenzhen LOH...,"The contract is valid for 5 years, but no spec...",Not specified in this contract.,The seller is liable for any damage and loss o...,success,NaN


In [7]:
# Summary statistics
print(df["status"].value_counts())
print(f"\nAverage summary length: {df['summary'].str.len().mean():.0f} chars")

status
success    50
Name: count, dtype: int64

Average summary length: 522 chars


In [8]:
# Display one result nicely
row = df.iloc[0]
print(f"Contract  : {row['contract_id']} – {row['title'][:60]}")
print(f"\nSUMMARY:\n{row['summary']}")
print(f"\nTERMINATION:\n{row['termination_clause']}")
print(f"\nCONFIDENTIALITY:\n{row['confidentiality_clause']}")
print(f"\nLIABILITY:\n{row['liability_clause']}")

Contract  : contract_0001 – LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT

SUMMARY:
This is a Distributor Agreement between Electric City Corp. and Electric City of Illinois LLC, outlining the terms and conditions for the distribution of Electric City's energy efficiency device, the Energy Saver, within the State of Illinois. Key obligations include the Distributor's exclusive right to sell and distribute the Products, the Company's obligation to sell and deliver Products to the Distributor, and the Distributor's obligation to promote the distribution, sales, and use of Products. Notable risks include the Distributor's potential liability for failure to meet minimum purchase expectations, and the Company's potential liability for failure to deliver Products on time, although liability for consequential or liquidated damages is limited unless specifically agreed upon in writing.

TERMINATION:
Either party may terminate this Distributor Agreement upon 10 years, with the option to r

## 4. (Bonus) Semantic Search over Clauses

In [9]:
# Install if needed: pip install sentence-transformers
from src.semantic_search import build_index, search, print_search_results

# Build embedding index from results
index = build_index(results)
print(f"Index built with {len(index['texts'])} clause segments")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Index built with 154 clause segments


In [10]:
# Example searches
queries = [
    "termination without cause 30 days notice",
    "indemnification and liability cap",
    "non-disclosure of confidential information",
]

for q in queries:
    hits = search(q, index, top_k=3)
    print_search_results(q, hits)


Query: 'termination without cause 30 days notice'

#1  Score: 0.7400
   Contract : contract_0004 – CENTRACKINTERNATIONALINC_10_29_1999-EX-10.3-WEB SITE HOSTING
   Field    : termination_clause
   Excerpt  : Either party may terminate this Agreement without cause at any time effective upon thirty (30) days' written notice.

#2  Score: 0.7322
   Contract : contract_0040 – CytodynInc_20200109_10-Q_EX-10.5_11941634_EX-10.5_License Ag
   Field    : termination_clause
   Excerpt  : Either party may terminate this Agreement upon 30 days written notice, subject to certain conditions and exceptions.

#3  Score: 0.7272
   Contract : contract_0010 – PACIRA PHARMACEUTICALS, INC. - A_R STRATEGIC LICENSING, DIST
   Field    : termination_clause
   Excerpt  : Either party may terminate this Agreement upon 30 days written notice, subject to certain conditions and obligations.

Query: 'indemnification and liability cap'

#1  Score: 0.5044
   Contract : contract_0030 – GLOBALTECHNOLOGIESLTD_06_08_2020-

## 5. Save Final Output

In [11]:
df.to_csv(OUTPUT_DIR / "results.csv", index=False)
print(f"Saved {len(df)} rows to {OUTPUT_DIR / 'results.csv'}")

with open(OUTPUT_DIR / "results.json", "w") as f:
    json.dump(results, f, indent=2)
print(f"Saved JSON to {OUTPUT_DIR / 'results.json'}")

Saved 50 rows to output\results.csv
Saved JSON to output\results.json
